In [ ]:
import wandb
import torch
from tqdm import tqdm
from ml_exp.utils import TFLOPS_TABLE


In [2]:
ENTITY = "gana-moharram"
TASK_NAME = "image_classification"
DATASET_NAME = "dermamnist"
PROJECT= f"{TASK_NAME}_{DATASET_NAME}"

In [ ]:
api = wandb.Api()
runs = api.runs(f"{ENTITY}/{PROJECT}")
runs_list = list(runs)
print("Runs Loaded!")

In [ ]:
calc_metrics = True
metrics = [("val_acc", "max"), ("val_auroc_weighted", "max"), ("val_loss", "min")]
calc_gpu_hours = True

for run in tqdm(runs_list):
    if calc_metrics:
        values = run.history()
        for METRIC_NAME, METRIC_TYPE in metrics:
            if METRIC_NAME not in run.summary or METRIC_NAME not in values.columns:
                if run == runs_list[0]:
                    # raise ValueError(f"Metric {METRIC_NAME} not found in first run {run.id}")
                    # print(f"Metric {METRIC_NAME} not found in first run {run.id}. Continuing...")
                    continue
                else:
                    # print(f"Metric {METRIC_NAME} not found in run {run.id}. Continuing...")
                    continue
            else:
                NEW_METRIC_NAME = f"{METRIC_TYPE}_{METRIC_NAME}"
                if NEW_METRIC_NAME in run.summary:
                    # print(f"Metric {NEW_METRIC_NAME} already exists in run {run.id}. Skipping...")
                    continue
                values[METRIC_NAME] = values[METRIC_NAME].astype(float)
                if METRIC_TYPE == "max":
                    best_value = values[METRIC_NAME].max(skipna=True)
                elif METRIC_TYPE == "min":
                    best_value = values[METRIC_NAME].min(skipna=True)
                else:
                    raise ValueError(f"Invalid metric type: {METRIC_TYPE}")
                
                if not np.isnan(best_value):
                    run.summary[NEW_METRIC_NAME] = best_value
                    run.summary[f"{NEW_METRIC_NAME}_epochnum"] = values[values[METRIC_NAME] == best_value]["epoch"].values[-1]

    if calc_gpu_hours:
        gpu_name = run.config.get("device")
        precision = run.config.get("trainer", {}).get("trainer", {}).get("precision")
        tflops = TFLOPS_TABLE.get(gpu_name, {}).get(precision, None)
        if tflops:
            try:
                duration_hours = run.summary["_runtime"] / 3600
            except KeyError:
                continue
            gpu_mean_util = run.history(stream="systemMetrics")["system.gpu.0.gpu"].mean() / 100
            tflop_hours = tflops * duration_hours * gpu_mean_util
            run.summary[f"time/tflop_hours_{gpu_name}_{precision}"] = tflop_hours
            run.summary[f"time/tflop_hours"] = tflop_hours
            
    run.summary.update()